# The AMOEBA force field

PBJ has support for the AMOEBA force field, which has been used previously in the context of PB in the work by [Schnieders et al](https://pubs.aip.org/aip/jcp/article-abstract/126/12/124114/937503/Polarizable-atomic-multipole-solutes-in-a-Poisson?redirectedFrom=fulltext) and [Cooper](https://onlinelibrary.wiley.com/doi/abs/10.1002/jcc.25820), where it was ported to the BEM code [PyGBe](https://github.com/pygbe/pygbe). Next, an example

In [ ]:
import pbj

In [2]:
simulation = pbj.implicit_solvent.Simulation(simulation_type='lpbe', force_field='amoeba')
simulation.ep_ex = 78.3

PBJ uses a SOR self-consistent scheme for the induced dipoles. You can access and modify the tolerance with `induced_dipole_iter_tol`

In [3]:
print(simulation.induced_dipole_iter_tol)

0.01


AMOEBA has several definitions of atomic radii. When generating the solute, `radius_keyword` can be `vdw` or `solute`, and `solute_radius_type` can be `PB`, `GK`, or `DDCOSMO` (depending on what was used to optimize those radiuses). As this is a PB calculation, we'll use `PB`for the `solute_radius_type`.

In [4]:
simulation.add_solute("1pgb_amoeba/1pgb.xyz", mesh_density=2, radius_keyword="vdw", solute_radius_type="PB", ep_in=1.)

Reading parameters from 1pgb_amoeba/amoebapro13.prm
Reading parameters from 1pgb_amoeba/amoebapro13.prm


 <<INFO>> Starting NanoShaper 0.7.8
 <<INFO>> Loading atoms....
 <<INFO>> Read 927 atoms
 <<INFO>> Geometric baricenter ->  8.91805 12.1305 18.5721
 <<INFO>> Grid is 45
 <<INFO>> MAX 28.918 32.1305 38.5721
 <<INFO>> MIN -11.082 -7.8695 -1.42795
 <<INFO>> Perfil 90 %
 <<INFO>> Rmaxdim 37.3799
 <<INFO>> Allocating memory...ok!
 <<INFO>> Initialization completed
 <<INFO>> Adjusting self intersection grid 
 <<INFO>> Self intersection grid is (before) 26
 <<INFO>> Self intersection grid is 16
 <<INFO>> Allocating self intersection grid....ok!
 <<INFO>> Computing alpha shape complex....ok!
 <<INFO>> Checking 115 probes for self intersections...ok!
 <<INFO>> Surface build-up time.. 0 [s]
 <<INFO>> Probe Radius value 1.4
 <<INFO>> Number of ses cells -> 2654
 <<INFO>> Number of del_point cells -> 426
 <<INFO>> Number of regular del_edge cells -> 1304
 <<INFO>> Number of singular del_edge c

In [5]:
simulation.get_info(save_log=False)

Simulation type: lpbe_amoeba
Formulation: direct
Force field: amoeba
----------------------------------------
solute name: 1pgb
number of vertices: 5804
number of elements: 11604
mesh density: 2
total area (Ang^2): 3023.9828697534194
dielectric solute (-): 1.0
dielectric solvent (-): 78.3
inverse Debye length (1/Ang): 0.125
operator ensembeler: dense
rhs_constructor: numpy
discrete_form_type: weak
gmres_tolerance: 1e-05
gmres_restart: 1000
gmres_max_iterations: 1000
----------------------------------------


The next command activates a verbose version to see how the calculation advances. 

In [6]:
pbj.implicit_solvent.simulation.bempp.api.enable_console_logging("info")

<StreamHandler stderr (INFO)>

And now we calculate...

In [7]:
simulation.calculate_solvation_energy()

/home/ian/miniconda3/envs/pbj/lib/python3.14/site-packages/bempp_cl/api/assembly/discrete_boundary_operator.py:619: SparseEfficiencyWarning: splu converted its input to CSC format
  solver = solver_interface(actual_mat)
bempp:HOST:INFO: OpenCL CPU Device set to: Intel(R) Core(TM) i5-8300H CPU @ 2.30GHz
/home/ian/miniconda3/envs/pbj/lib/python3.14/site-packages/pyopencl/__init__.py:578: UserWarning: PyOpenCL compiler caching failed with an exception:
[begin exception]
Traceback (most recent call last):
  File "/home/ian/miniconda3/envs/pbj/lib/python3.14/site-packages/pyopencl/cache.py", line 514, in create_built_program_from_source_cached
    _create_built_program_from_source_cached(
    ~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~^
            ctx, src, options_bytes, devices, cache_dir,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
            include_path=include_path)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/ian/miniconda3/envs/pbj/lib/python3.14/site-packages/py

In [13]:
simulation.get_info(save_log=False)

Simulation type: lpbe_amoeba
Formulation: direct
Force field: amoeba
----------------------------------------
solute name: 1pgb
number of vertices: 5804
number of elements: 11604
mesh density: 2
total area (Ang^2): 3023.9828697534194
dielectric solute (-): 1.0
dielectric solvent (-): 78.3
inverse Debye length (1/Ang): 0.125
operator ensembeler: dense
rhs_constructor: numpy
discrete_form_type: weak
gmres_tolerance: 1e-05
gmres_restart: 1000
gmres_max_iterations: 1000
----------------------------------------


In [12]:
simulation.get_results(save_log=False)

simulation type: lpbe_amoeba
formulation: direct
force field: amoeba
----------------------------------------
solute name: 1pgb
dielectric solute (-): 1.0
dielectric solvent (-): 78.3
inverse Debye length (1/Ang): 0.125
electrostatic solvation energy (kcal/mol): -897.460
----------------------------------------


We can explore a lot of the intermediate results, which are available in the results dictionary

In [9]:
print(simulation.solutes[0].results.keys())

dict_keys(['induced_dipole', 'phi', 'd_phi', 'gradphir_charges', 'd_phi_coulomb_multipole', 'phir_charges', 'gradgradphir_charges', 'phi_perm_multipoles', 'gradphi_perm_multipoles', 'gradgradphi_perm_multipoles', 'phi_induced_dipole_dissolved', 'gradphi_induced_dipole_dissolved', 'gradgradphi_induced_dipole_dissolved', 'induced_dipole_vacuum', 'dipole_iter_count_vacuum', 'phi_induced_dipole_vacuum', 'gradphi_induced_dipole_vacuum', 'gradgradphi_induced_dipole_vacuum', 'electrostatic_solvation_energy', 'coulomb_energy_dissolved', 'coulomb_energy_vacuum', 'electrostatic_solvation_energy_units'])


For example, the reaction field (derivative of the reaction potential) at the location of the atoms

In [10]:
print(simulation.solutes[0].results['gradphir_charges'])

[[-0.00205518 -0.00078725 -0.00477305]
 [-0.00149738 -0.00142422 -0.00350124]
 [-0.0016896  -0.00013359 -0.00250373]
 ...
 [-0.00217112 -0.00269777 -0.00192573]
 [-0.00208841 -0.00592403 -0.00400245]
 [-0.00328321 -0.00081042 -0.00352217]]


And even plot the surface potential

In [11]:
simulation.plot_surface_values(savefig=False)

Plotting solutes ['1pgb']
